# N9 Strip Architecture Sweep

Runs all architecture variants on the 9-node strip data using the same core model/training settings as `2d_strip.ipynb`. All outputs are collected under one `OUTPUT_DIR`, with per-architecture plots, energy landscapes, paper-ready comparison plots, and post-training Hessian diagnostics.

In [1]:
import os
from pathlib import Path

import jax
import jax.numpy as jnp

jax.config.update("jax_enable_x64", True)
os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")

from properties import StripN9Properties
from run_architectures import SweepConfig, run_architecture_sweep

# ---------------------------------------------------------
# Dataset/properties/model/training settings copied from
# 2d_strip.ipynb.
# ---------------------------------------------------------
ROOT = Path.cwd()
OUTPUT_DIR = ROOT / "arch_sweep_outputs_n9_strip_all_architectures"
PAPER_PLOT_DIR = OUTPUT_DIR / "paper_ready_architecture_comparison"

# train_file = "../experiment_data/n9_strip_train_dataset.npz"
train_file = "../experiment_data/strip_data/9_noded/n7_traj2_strip_dataset_14_pts.npz"
valid_file = "../experiment_data/n9_strip_test_dataset.npz"

properties = StripN9Properties(mass=0.001)
K_init_diag = (0.002, 0.005)
K_init_chol = (0.002, 0.0, 0.005)

cfg = SweepConfig(
    der_K_diag=K_init_diag,
    der_K_chol=K_init_chol,
    hidden=(10,),
    corr_factor=0.01,
    input_mode="raw",
    only_stretching_NN=False,
    only_bending_NN=False,
    zero_reference=True,
    activation="tanh",
    n_epochs=500,
    lr=1e-2,
    seed=42,
    valid_every=10,
    max_dlambda=1e-2,
    iters=20,
    ls_steps=10,
    abs_tol=1e-4,
    rel_tol=1e-4,
    early_stop=True,
    train_fail_on_nonconvergence=True,
    prediction_fail_on_nonconvergence=False,
    hessian_reg_strength=1e-6,
    hessian_reg_probes=1,
    hessian_reg_seed=0,
    force_key=None,
    force_loss_strength=0.0,
    force_components=(0, 1, 2),
    force_sign=1.0,
    return_loss_components=False,
    early_stopping=True,
    early_stopping_patience=200,
    early_stopping_min_delta=1e-5,
    restore_best_model=True,
    output_dir=str(OUTPUT_DIR),
    save_npz=True,
    save_model=True,
    save_plots=True,
    save_force_predictions=False,
    plot_force_predictions=False,
    save_hessian_diagnostics=False,  # post-process in the Hessian cell below
    save_energy_landscapes=False,  # generate these post-training in the cell below
    energy_snapshot_initial=True,
    energy_snapshot_final=True,
    energy_snapshot_epochs=(),
    energy_snapshot_every=None,
    energy_snapshot_use_valid=True,
    energy_snapshot_dpi=180,
    energy_snapshot_n_grid=None,
    verbose=True,
    continue_on_failure=True,
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Saving all results under: {OUTPUT_DIR.resolve()}")

Saving all results under: /Users/radha/GitRepos/dismech-jax/examples/slinky/slinky_2D/arch_sweep_outputs_n9_strip_all_architectures


In [2]:
from run_architectures import subset_all

# All registered architecture variants.
selected_architectures = subset_all()

# For a quicker debug pass, temporarily uncomment a smaller list:
# selected_architectures = [
#     "diag_energy_baseline",
#     "diag_energy_mlp",
#     "chol_energy_baseline",
# ]

selected_architectures = [
    arch for arch in selected_architectures
    if arch not in ["chol_stiffness_signed_mlp", "diag_stiffness_icnn", "chol_stiffness_icnn", "chol_stiffness_signed_icnn"]
]


print(f"Running {len(selected_architectures)} architectures:")
for name in selected_architectures:
    print(f"  - {name}")

print("\nPer-architecture plots:", cfg.save_plots)
print("Energy landscape snapshots during training:", cfg.save_energy_landscapes)
print("Force prediction plots:", cfg.plot_force_predictions)

Running 10 architectures:
  - diag_energy_baseline
  - diag_energy_mlp
  - diag_energy_icnn
  - mlp_energy
  - icnn_energy
  - chol_energy_baseline
  - chol_energy_mlp
  - chol_energy_icnn
  - diag_stiffness_mlp
  - chol_stiffness_mlp

Per-architecture plots: True
Energy landscape snapshots during training: False
Force prediction plots: False


In [3]:
results = run_architecture_sweep(
    properties=properties,
    train_file=train_file,
    valid_file=valid_file,
    cfg=cfg,
    selected_architectures=selected_architectures,
)

successes = [name for name, result in results.items() if result["success"]]
failures = {name: result["failure_reason"] for name, result in results.items() if not result["success"]}

print(f"Succeeded: {len(successes)}/{len(results)}")
if failures:
    print("Failures:")
    for name, reason in failures.items():
        print(f"  - {name}: {reason}")
else:
    print("No architecture failures.")

Running architecture: diag_energy_baseline
  model_cls               : DiagonalPlusEnergyNN
  which_case              : baseline
  hidden                  : (10,)
  input_mode              : raw
  only_stretching_NN      : False
  only_bending_NN         : False
  activation              : tanh
  corr_factor             : 0.01
  zero_reference          : True
  seed                    : 42
  max_dlambda             : 0.01
  iters                   : 20
  ls_steps                : 10
  abs_tol                 : 0.0001
  rel_tol                 : 0.0001
  early_stop              : True
  training fail_on_nonconvergence       : True
  validation loss fail_on_nonconvergence: False
  prediction fail_on_nonconvergence     : False
  early_stopping          : True
  early_stopping_patience : 200
  restore_best_model      : True
  hessian_reg_strength    : 1e-06
  hessian_reg_probes      : 1
  hessian_reg_seed        : 0
  force_key               : None
  force_loss_strength     : 0.0
  force_c

/Users/radha/GitRepos/dismech-jax/examples/slinky/slinky_2D/architecture_plots.py:234: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig.tight_layout()


Running architecture: diag_energy_mlp
  model_cls               : DiagonalPlusEnergyNN
  which_case              : MLP
  hidden                  : (10,)
  input_mode              : raw
  only_stretching_NN      : False
  only_bending_NN         : False
  activation              : tanh
  corr_factor             : 0.01
  zero_reference          : True
  seed                    : 42
  max_dlambda             : 0.01
  iters                   : 20
  ls_steps                : 10
  abs_tol                 : 0.0001
  rel_tol                 : 0.0001
  early_stop              : True
  training fail_on_nonconvergence       : True
  validation loss fail_on_nonconvergence: False
  prediction fail_on_nonconvergence     : False
  early_stopping          : True
  early_stopping_patience : 200
  restore_best_model      : True
  hessian_reg_strength    : 1e-06
  hessian_reg_probes      : 1
  hessian_reg_seed        : 0
  force_key               : None
  force_loss_strength     : 0.0
  force_components 

In [4]:
from run_architectures import generate_energy_landscapes_for_sweep

# Recreate energy landscapes from saved config.json, model.eqx, and dataset paths.
# This avoids doing landscape solves/plots inside the training loop.
energy_landscape_paths = generate_energy_landscapes_for_sweep(
    str(OUTPUT_DIR),
    architectures=successes if "successes" in globals() else selected_architectures,
    include_initial=False,
    include_final=True,
    use_valid=cfg.energy_snapshot_use_valid,
    traj_idx=cfg.energy_landscape_spec.traj_idx,
    dpi=cfg.energy_snapshot_dpi,
    n_grid=cfg.energy_snapshot_n_grid,
    continue_on_failure=True,
)

print("Energy landscape plots regenerated for", len(energy_landscape_paths), "runs.")

[ok] chol_energy_baseline: {'final': '/Users/radha/GitRepos/dismech-jax/examples/slinky/slinky_2D/arch_sweep_outputs_n9_strip_all_architectures/chol_energy_baseline__hid_10__inp_raw__stretchNN_0__bendNN_0__act_tanh__corr_0.01__zr1__seed_42__mdl_0.01__it_20__hreg_1e-06__hprobe_1__hseed_0/energy_landscapes/energy_landscape_final.png'}
[ok] chol_energy_icnn: {'final': '/Users/radha/GitRepos/dismech-jax/examples/slinky/slinky_2D/arch_sweep_outputs_n9_strip_all_architectures/chol_energy_icnn__hid_10__inp_raw__stretchNN_0__bendNN_0__act_tanh__corr_0.01__zr1__seed_42__mdl_0.01__it_20__hreg_1e-06__hprobe_1__hseed_0/energy_landscapes/energy_landscape_final.png'}
[ok] chol_stiffness_mlp: {'final': '/Users/radha/GitRepos/dismech-jax/examples/slinky/slinky_2D/arch_sweep_outputs_n9_strip_all_architectures/chol_stiffness_mlp__hid_10__inp_raw__stretchNN_0__bendNN_0__act_tanh__corr_0.01__zr1__seed_42__mdl_0.01__it_20__hreg_1e-06__hprobe_1__hseed_0/energy_landscapes/energy_landscape_final.png'}
[ok] di

In [5]:
from architecture_plots import plot_architecture_comparison_paper, plot_summary_final_losses

PAPER_PLOT_DIR.mkdir(parents=True, exist_ok=True)

# Paper-ready PDF comparisons across all successfully completed architectures.
paper_architectures = successes if "successes" in globals() else selected_architectures
paper_paths = plot_architecture_comparison_paper(
    architectures=paper_architectures,
    results_dir=str(OUTPUT_DIR),
    output_dir=str(PAPER_PLOT_DIR),
    traj_idx=0,
)

# Convenience PNG summary of final losses from the in-memory results dict.
plot_summary_final_losses(
    {name: results[name] for name in paper_architectures},
    save_path=str(PAPER_PLOT_DIR / "final_loss_summary.png"),
    show=False,
)

print("Paper-ready plots written to:", PAPER_PLOT_DIR.resolve())
for key, path in paper_paths.items():
    if key != "colors":
        print(f"  {key}: {path}")

Paper-ready plots written to: /Users/radha/GitRepos/dismech-jax/examples/slinky/slinky_2D/arch_sweep_outputs_n9_strip_all_architectures/paper_ready_architecture_comparison
  training_loss: /Users/radha/GitRepos/dismech-jax/examples/slinky/slinky_2D/arch_sweep_outputs_n9_strip_all_architectures/paper_ready_architecture_comparison/training_loss_comparison.pdf
  validation_loss: /Users/radha/GitRepos/dismech-jax/examples/slinky/slinky_2D/arch_sweep_outputs_n9_strip_all_architectures/paper_ready_architecture_comparison/validation_loss_comparison.pdf
  training_trajectory: /Users/radha/GitRepos/dismech-jax/examples/slinky/slinky_2D/arch_sweep_outputs_n9_strip_all_architectures/paper_ready_architecture_comparison/training_trajectory_comparison.pdf
  validation_trajectory: /Users/radha/GitRepos/dismech-jax/examples/slinky/slinky_2D/arch_sweep_outputs_n9_strip_all_architectures/paper_ready_architecture_comparison/validation_trajectory_comparison.pdf
  training_xz_trajectory: /Users/radha/GitRe

In [6]:
import subprocess
import sys

# Post-training Hessian diagnostics. This is intentionally separated from
# training so the architecture sweep stays fast.
HESSIAN_USE_PREDICTED = True
HESSIAN_STRIDE = 10
HESSIAN_MAX_TRAJECTORIES = 1  # set to None to process all trajectories
HESSIAN_SPLITS = ("train", "valid")

cmd = [
    sys.executable,
    "compute_architecture_hessian_diagnostics.py",
    str(OUTPUT_DIR),
    "--stride",
    str(HESSIAN_STRIDE),
    "--splits",
    *HESSIAN_SPLITS,
]
if HESSIAN_USE_PREDICTED:
    cmd.append("--use-predicted")
if HESSIAN_MAX_TRAJECTORIES is None:
    cmd.append("--all-trajectories")
else:
    cmd.extend(["--max-trajectories", str(HESSIAN_MAX_TRAJECTORIES)])

print("Running:", " ".join(cmd))
subprocess.run(cmd, cwd=str(ROOT), check=True)
print("Hessian diagnostics written into each architecture directory under:", OUTPUT_DIR.resolve())

Running: /Users/radha/GitRepos/dismech-jax/.venv/bin/python compute_architecture_hessian_diagnostics.py /Users/radha/GitRepos/dismech-jax/examples/slinky/slinky_2D/arch_sweep_outputs_n9_strip_all_architectures --stride 10 --splits train valid --use-predicted --max-trajectories 1
[ok] /Users/radha/GitRepos/dismech-jax/examples/slinky/slinky_2D/arch_sweep_outputs_n9_strip_all_architectures/chol_energy_baseline__hid_10__inp_raw__stretchNN_0__bendNN_0__act_tanh__corr_0.01__zr1__seed_42__mdl_0.01__it_20__hreg_1e-06__hprobe_1__hseed_0 | train: M=8.620e+01, kappa=1.459e+03, states=2 | valid: M=2.825e+01, kappa=4.577e+02, states=1
[ok] /Users/radha/GitRepos/dismech-jax/examples/slinky/slinky_2D/arch_sweep_outputs_n9_strip_all_architectures/chol_energy_icnn__hid_10__inp_raw__stretchNN_0__bendNN_0__act_tanh__corr_0.01__zr1__seed_42__mdl_0.01__it_20__hreg_1e-06__hprobe_1__hseed_0 | train: M=2.715e+01, kappa=3.928e+02, states=2 | valid: M=1.634e+01, kappa=3.199e+02, states=1
[skip] /Users/radha/Gi

In [7]:
print("All artifacts are organized under:")
print(" ", OUTPUT_DIR.resolve())
print("\nMain subfolders/files to inspect:")
print("  - <architecture>/results.npz")
print("  - <architecture>/model.eqx")
print("  - <architecture>/loss_curves.png")
print("  - <architecture>/pred_vs_truth_*")
print("  - <architecture>/energy_landscapes/")
print("  - <architecture>/hessian_diagnostics_*.npz")
print("  - <architecture>/hessian_diagnostics_summary.json")
print("  - paper_ready_architecture_comparison/*.pdf")

All artifacts are organized under:
  /Users/radha/GitRepos/dismech-jax/examples/slinky/slinky_2D/arch_sweep_outputs_n9_strip_all_architectures

Main subfolders/files to inspect:
  - <architecture>/results.npz
  - <architecture>/model.eqx
  - <architecture>/loss_curves.png
  - <architecture>/pred_vs_truth_*
  - <architecture>/energy_landscapes/
  - <architecture>/hessian_diagnostics_*.npz
  - <architecture>/hessian_diagnostics_summary.json
  - paper_ready_architecture_comparison/*.pdf
